# 07A FORCE robust RF nested grouped candidate selection

> Saved outputs are from an earlier local run. Portable path edits have **not** been re-executed. See the repository README for setup and scientific limitations.


In [1]:
import sys
import subprocess
import importlib.util

required_packages = {
    "pandas": "pandas",
    "numpy": "numpy",
    "scikit-learn": "sklearn",
    "joblib": "joblib",
    "pyarrow": "pyarrow",
}

for package_name, module_name in required_packages.items():
    if importlib.util.find_spec(module_name) is None:
        print(f"Installing {package_name} ...")
        subprocess.check_call(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                package_name,
            ]
        )

print("All required packages are available.")

All required packages are available.


In [2]:
from pathlib import Path
from collections import defaultdict
import gc
import hashlib
import json
import platform
import time
import warnings

import joblib
import numpy as np
import pandas as pd
import sklearn

from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    log_loss,
)

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 300)
pd.set_option("display.max_rows", 300)
pd.set_option("display.max_colwidth", 300)

RANDOM_STATE = 142
PROBABILITY_EPSILON = 1e-12
SELECTION_STRESS_WEIGHT = 0.5
AUGMENTATION_FRACTION = 0.20
PREDICTION_BATCH_SIZE = 100_000

GLOBAL_CLASS_CODES = np.array(
    [
        30000,
        65000,
        65030,
        70000,
        70032,
        74000,
        80000,
        86000,
        88000,
        90000,
        93000,
        99000,
    ],
    dtype=np.int64,
)

CLASS_CODE_TO_INDEX = {
    int(code): index
    for index, code in enumerate(GLOBAL_CLASS_CODES)
}

PROBABILITY_COLUMNS = [
    f"prob_{code}"
    for code in GLOBAL_CLASS_CODES
]

np.random.seed(RANDOM_STATE)

In [3]:
RUN_MODE = "full"  # "pilot" or "full"

ACTIVE_OUTER_FOLDS = (
    [1]
    if RUN_MODE == "pilot"
    else [1, 2, 3, 4, 5]
)

ALL_FOLDS = [1, 2, 3, 4, 5]

if RUN_MODE == "pilot":
    INNER_TRAINING_ROW_CAP = 120_000
    INNER_N_ESTIMATORS = 120
    FINAL_N_ESTIMATORS = 300
else:
    INNER_TRAINING_ROW_CAP = 250_000
    INNER_N_ESTIMATORS = 250
    FINAL_N_ESTIMATORS = 600

SKIP_COMPLETED_OUTER_FOLDS = True
SKIP_COMPLETED_INNER_SEARCH = True
JOBLIB_COMPRESSION = 1

# Portable local paths. Override upstream data/artifacts with environment variables.
import os

def _find_repo_root():
    start = Path(os.environ.get("STRATASHIFT_REPO_ROOT", Path.cwd())).expanduser().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "README.md").is_file() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError("StrataShift root not found. Open the repository in Jupyter or set STRATASHIFT_REPO_ROOT.")

REPO_ROOT = _find_repo_root()
TRAIN_PATH = Path(os.environ.get("STRATASHIFT_TRAIN_CSV", str(REPO_ROOT / "data" / "raw" / "train.csv"))).expanduser().resolve()
PROJECT_ROOT = Path(os.environ.get("STRATASHIFT_ARTIFACT_ROOT", str(REPO_ROOT / "artifacts"))).expanduser().resolve()


FOLD_ASSIGNMENT_PATH = (
    PROJECT_ROOT
    / "01_clean_splits_support"
    / "10_grouped_5fold_well_assignment.csv"
)

STAGE_05A_DIR = (
    PROJECT_ROOT
    / "05A_raw_log_feature_contract"
    / "full"
)

PRIMARY_CONTRACT_PATH = (
    STAGE_05A_DIR
    / "10_primary_raw_feature_contract.json"
)

STAGE_05A_GATE_PATH = (
    STAGE_05A_DIR
    / "13_raw_feature_contract_gate.json"
)

STAGE_05B_DIR = (
    PROJECT_ROOT
    / "05B_raw_log_RF_grouped_well_CV"
    / "full"
)

BASELINE_OOF_PATH = (
    STAGE_05B_DIR
    / "16_OOF_predictions_raw8.parquet"
)

BASELINE_GATE_PATH = (
    STAGE_05B_DIR
    / "17_stage_gate.json"
)

STAGE_06C_DIR = (
    PROJECT_ROOT
    / "06C_raw_vs_legacy_representation_amplification"
    / "full"
)

STAGE_06C_GATE_PATH = (
    STAGE_06C_DIR
    / "14_stage_gate.json"
)

STAGE_06C_CONFIG_PATH = (
    STAGE_06C_DIR
    / "15_run_configuration.json"
)

OUTPUT_DIR = (
    PROJECT_ROOT
    / "07A_robust_RF_nested_grouped_candidate_selection"
    / RUN_MODE
)

MODEL_DIR = (
    OUTPUT_DIR
    / "models"
)

OUTER_OOF_DIR = (
    OUTPUT_DIR
    / "outer_oof"
)

INNER_SEARCH_DIR = (
    OUTPUT_DIR
    / "inner_search"
)

for directory in [
    OUTPUT_DIR,
    MODEL_DIR,
    OUTER_OOF_DIR,
    INNER_SEARCH_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

required_paths = [
    TRAIN_PATH,
    FOLD_ASSIGNMENT_PATH,
    PRIMARY_CONTRACT_PATH,
    STAGE_05A_GATE_PATH,
    BASELINE_OOF_PATH,
    BASELINE_GATE_PATH,
    STAGE_06C_GATE_PATH,
    STAGE_06C_CONFIG_PATH,
]

missing_paths = [
    str(path)
    for path in required_paths
    if not path.exists()
]

if missing_paths:
    raise FileNotFoundError(
        "Missing required inputs:\n"
        + "\n".join(missing_paths)
    )

print("RUN_MODE:", RUN_MODE)
print("Outer folds:", ACTIVE_OUTER_FOLDS)
print("Inner row cap:", INNER_TRAINING_ROW_CAP)
print("Inner trees:", INNER_N_ESTIMATORS)
print("Final trees:", FINAL_N_ESTIMATORS)
print("Output:", OUTPUT_DIR)

RUN_MODE: full
Outer folds: [1, 2, 3, 4, 5]
Inner row cap: 250000
Inner trees: 250
Final trees: 600
Output: <LOCAL_PATH_REDACTED>


# 2. Candidate registry

In [4]:
RAW8_FEATURES = [
    "GR",
    "NPHI",
    "RHOB",
    "DTC",
    "RDEP",
    "CALI",
    "PEF",
    "SP",
]

CORE6_FEATURES = [
    "GR",
    "NPHI",
    "RHOB",
    "DTC",
    "RDEP",
    "CALI",
]

CANDIDATES = {
    "regularized_raw8": {
        "features": RAW8_FEATURES,
        "augmentation": "none",
        "augmentation_features": [],
    },
    "regularized_core6": {
        "features": CORE6_FEATURES,
        "augmentation": "none",
        "augmentation_features": [],
    },
    "global_single_log_dropout_raw8": {
        "features": RAW8_FEATURES,
        "augmentation": "single_log_median_replacement",
        "augmentation_features": RAW8_FEATURES,
    },
    "targeted_DTC_RHOB_dropout_raw8": {
        "features": RAW8_FEATURES,
        "augmentation": "single_log_median_replacement",
        "augmentation_features": [
            "DTC",
            "RHOB",
        ],
    },
}

BASE_RF_PARAMETERS = {
    "max_depth": 60,
    "min_samples_split": 10,
    "min_samples_leaf": 5,
    "max_features": "sqrt",
    "bootstrap": True,
    "max_samples": 0.75,
    "class_weight": "balanced_subsample",
    "criterion": "gini",
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
}

candidate_registry_rows = []

for candidate_id, specification in CANDIDATES.items():
    candidate_registry_rows.append({
        "candidate_id": candidate_id,
        "feature_count": len(
            specification[
                "features"
            ]
        ),
        "features": "|".join(
            specification[
                "features"
            ]
        ),
        "augmentation": specification[
            "augmentation"
        ],
        "augmentation_features": "|".join(
            specification[
                "augmentation_features"
            ]
        ),
        "augmentation_fraction": (
            AUGMENTATION_FRACTION
            if specification[
                "augmentation"
            ]
            != "none"
            else 0.0
        ),
        "single_log_at_a_time": True,
        "label_unchanged": True,
        "inner_training_only_statistics": True,
    })

candidate_registry = pd.DataFrame(
    candidate_registry_rows
)

display(candidate_registry)

,candidate_id,feature_count,features,augmentation,augmentation_features,augmentation_fraction,single_log_at_a_time,label_unchanged,inner_training_only_statistics
0,regularized_raw8,8,GR|NPHI|RHOB|DTC|RDEP|CALI|PEF|SP,none,,0.0,True,True,True
1,regularized_core6,6,GR|NPHI|RHOB|DTC|RDEP|CALI,none,,0.0,True,True,True
2,global_single_log_dropout_raw8,8,GR|NPHI|RHOB|DTC|RDEP|CALI|PEF|SP,single_log_median_replacement,GR|NPHI|RHOB|DTC|RDEP|CALI|PEF|SP,0.2,True,True,True
3,targeted_DTC_RHOB_dropout_raw8,8,GR|NPHI|RHOB|DTC|RDEP|CALI|PEF|SP,single_log_median_replacement,DTC|RHOB,0.2,True,True,True


In [5]:
def sha256_file(
    path,
    chunk_size=1024 * 1024,
):
    digest = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as file:
        while True:
            chunk = file.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


def stable_seed(*parts):
    text = "||".join(
        str(part)
        for part in parts
    )

    return int(
        hashlib.sha256(
            text.encode("utf-8")
        ).hexdigest()[:8],
        16,
    )


def json_safe(value):
    if isinstance(value, dict):
        return {
            str(key): json_safe(item)
            for key, item in value.items()
        }

    if isinstance(value, (list, tuple)):
        return [
            json_safe(item)
            for item in value
        ]

    if isinstance(value, np.ndarray):
        return value.tolist()

    if isinstance(value, np.generic):
        return value.item()

    if isinstance(value, Path):
        return str(value)

    if (
        isinstance(value, float)
        and not np.isfinite(value)
    ):
        return None

    return value


def detect_csv_format(
    file_path,
    num_lines=5,
):
    for encoding in [
        "utf-8",
        "latin1",
        "iso-8859-1",
        "cp1252",
    ]:
        try:
            with open(
                file_path,
                "r",
                encoding=encoding,
            ) as file:
                lines = [
                    file.readline()
                    for _ in range(
                        num_lines
                    )
                ]

            if any(
                "," in line
                for line in lines
            ):
                separator = ","

            elif any(
                ";" in line
                for line in lines
            ):
                separator = ";"

            elif any(
                "\t" in line
                for line in lines
            ):
                separator = "\t"

            else:
                separator = ","

            return {
                "sep": separator,
                "encoding": encoding,
            }

        except UnicodeDecodeError:
            continue

    return {
        "sep": ",",
        "encoding": "utf-8",
    }


def align_probabilities(
    local_probabilities,
    local_classes,
):
    local_probabilities = np.asarray(
        local_probabilities,
        dtype=np.float64,
    )

    aligned = np.zeros(
        (
            local_probabilities.shape[0],
            len(
                GLOBAL_CLASS_CODES
            ),
        ),
        dtype=np.float64,
    )

    for local_index, class_code in enumerate(
        local_classes
    ):
        aligned[
            :,
            CLASS_CODE_TO_INDEX[
                int(
                    class_code
                )
            ],
        ] = local_probabilities[
            :,
            local_index,
        ]

    row_sums = aligned.sum(
        axis=1,
        keepdims=True,
    )

    if np.any(
        row_sums <= 0
    ):
        raise RuntimeError(
            "Probability row with non-positive sum."
        )

    return (
        aligned
        / row_sums
    )


def predict_probabilities_batched(
    model,
    matrix,
    batch_size=PREDICTION_BATCH_SIZE,
):
    probability_parts = []

    for start in range(
        0,
        len(matrix),
        batch_size,
    ):
        stop = min(
            start + batch_size,
            len(matrix),
        )

        local = model.predict_proba(
            matrix[
                start:stop
            ]
        )

        probability_parts.append(
            align_probabilities(
                local,
                model.classes_,
            )
        )

    return np.vstack(
        probability_parts
    )


def row_outputs(
    y_true,
    probabilities,
):
    y_true = np.asarray(
        y_true,
        dtype=np.int64,
    )

    probabilities = np.asarray(
        probabilities,
        dtype=np.float64,
    )

    true_indices = np.array([
        CLASS_CODE_TO_INDEX[
            int(code)
        ]
        for code in y_true
    ])

    true_probability = probabilities[
        np.arange(
            len(
                y_true
            )
        ),
        true_indices,
    ]

    row_log_loss = -np.log(
        np.clip(
            true_probability,
            PROBABILITY_EPSILON,
            1.0,
        )
    )

    predicted = GLOBAL_CLASS_CODES[
        np.argmax(
            probabilities,
            axis=1,
        )
    ]

    return {
        "true_probability": true_probability,
        "row_log_loss": row_log_loss,
        "predicted": predicted,
        "correct": predicted == y_true,
    }


def well_balanced_mean(
    values,
    wells,
):
    frame = pd.DataFrame({
        "WELL": np.asarray(
            wells
        ),
        "value": np.asarray(
            values,
            dtype=np.float64,
        ),
    })

    return float(
        frame.groupby(
            "WELL"
        )[
            "value"
        ]
        .mean()
        .mean()
    )


def prediction_metrics(
    y_true,
    probabilities,
    wells,
):
    outputs = row_outputs(
        y_true,
        probabilities,
    )

    per_well = pd.DataFrame({
        "WELL": np.asarray(
            wells
        ),
        "correct": outputs[
            "correct"
        ].astype(float),
        "row_log_loss": outputs[
            "row_log_loss"
        ],
    }).groupby(
        "WELL",
        as_index=False,
    ).agg(
        accuracy=(
            "correct",
            "mean",
        ),
        log_loss=(
            "row_log_loss",
            "mean",
        ),
    )

    predicted = outputs[
        "predicted"
    ]

    return {
        "rows": int(
            len(
                y_true
            )
        ),
        "wells": int(
            per_well[
                "WELL"
            ].nunique()
        ),
        "accuracy": float(
            accuracy_score(
                y_true,
                predicted,
            )
        ),
        "balanced_accuracy": float(
            balanced_accuracy_score(
                y_true,
                predicted,
            )
        ),
        "macro_f1": float(
            f1_score(
                y_true,
                predicted,
                labels=GLOBAL_CLASS_CODES,
                average="macro",
                zero_division=0,
            )
        ),
        "log_loss": float(
            log_loss(
                y_true,
                probabilities,
                labels=GLOBAL_CLASS_CODES,
            )
        ),
        "well_balanced_accuracy": float(
            per_well[
                "accuracy"
            ].mean()
        ),
        "well_balanced_log_loss": float(
            per_well[
                "log_loss"
            ].mean()
        ),
        "worst_well_log_loss": float(
            per_well[
                "log_loss"
            ].max()
        ),
    }


def clean_feature_series(
    dataframe,
    feature,
):
    values = pd.to_numeric(
        dataframe[
            feature
        ],
        errors="coerce",
    ).astype(float)

    if feature == "SP":
        sentinel_mask = np.isclose(
            values.to_numpy(float),
            -999.0,
            rtol=0.0,
            atol=1e-12,
            equal_nan=False,
        )

        values.loc[
            sentinel_mask
        ] = np.nan

    nonfinite = ~np.isfinite(
        values.to_numpy(float)
    )

    values.loc[
        nonfinite
    ] = np.nan

    if feature == "RDEP":
        nonpositive = (
            values.notna()
            & (
                values
                <= 0
            )
        )

        values.loc[
            nonpositive
        ] = np.nan

    return values.astype(
        np.float32
    )


def well_balanced_training_sample(
    dataframe,
    mask,
    max_rows,
    seed,
):
    eligible = dataframe.loc[
        mask,
        [
            "ORIGINAL_ROW_ID",
            "WELL",
            "LITHOLOGY_CODE",
        ],
    ].copy()

    if len(
        eligible
    ) <= max_rows:
        return eligible.index.to_numpy()

    rng = np.random.default_rng(
        seed
    )

    wells = eligible[
        "WELL"
    ].unique()

    per_well_cap = max(
        1,
        int(
            np.ceil(
                max_rows
                / len(
                    wells
                )
            )
        ),
    )

    sampled_indices = []

    for _, well_frame in eligible.groupby(
        "WELL",
        sort=False,
    ):
        local_indices = well_frame.index.to_numpy()

        if len(
            local_indices
        ) > per_well_cap:
            local_indices = rng.choice(
                local_indices,
                size=per_well_cap,
                replace=False,
            )

        sampled_indices.extend(
            local_indices.tolist()
        )

    sampled_indices = np.array(
        sampled_indices,
        dtype=np.int64,
    )

    if len(
        sampled_indices
    ) > max_rows:
        sampled_indices = rng.choice(
            sampled_indices,
            size=max_rows,
            replace=False,
        )

    sampled_classes = set(
        dataframe.loc[
            sampled_indices,
            "LITHOLOGY_CODE",
        ].astype(int)
    )

    required_classes = set(
        dataframe.loc[
            mask,
            "LITHOLOGY_CODE",
        ].astype(int)
    )

    missing_classes = sorted(
        required_classes
        - sampled_classes
    )

    additions = []

    for class_code in missing_classes:
        class_indices = dataframe.index[
            mask
            & (
                dataframe[
                    "LITHOLOGY_CODE"
                ]
                == class_code
            )
        ].to_numpy()

        additions.append(
            int(
                rng.choice(
                    class_indices
                )
            )
        )

    if additions:
        sampled_indices = np.unique(
            np.concatenate([
                sampled_indices,
                np.array(
                    additions,
                    dtype=np.int64,
                ),
            ])
        )

    return sampled_indices


def build_augmented_training_matrix(
    matrix,
    labels,
    feature_names,
    candidate_specification,
    feature_medians,
    seed,
):
    matrix = np.asarray(
        matrix,
        dtype=np.float32,
    )

    labels = np.asarray(
        labels,
        dtype=np.int64,
    )

    if (
        candidate_specification[
            "augmentation"
        ]
        == "none"
    ):
        return (
            matrix,
            labels,
            0,
        )

    rng = np.random.default_rng(
        seed
    )

    augmented_count = max(
        1,
        int(
            round(
                len(
                    matrix
                )
                * AUGMENTATION_FRACTION
            )
        ),
    )

    selected_rows = rng.choice(
        len(
            matrix
        ),
        size=augmented_count,
        replace=False,
    )

    augmented_matrix = matrix[
        selected_rows
    ].copy()

    augmented_labels = labels[
        selected_rows
    ].copy()

    allowed_features = candidate_specification[
        "augmentation_features"
    ]

    allowed_positions = [
        feature_names.index(
            feature
        )
        for feature in allowed_features
    ]

    selected_feature_positions = rng.choice(
        allowed_positions,
        size=augmented_count,
        replace=True,
    )

    medians = np.asarray(
        feature_medians,
        dtype=np.float32,
    )

    augmented_matrix[
        np.arange(
            augmented_count
        ),
        selected_feature_positions,
    ] = medians[
        selected_feature_positions
    ]

    combined_matrix = np.concatenate(
        [
            matrix,
            augmented_matrix,
        ],
        axis=0,
    )

    combined_labels = np.concatenate(
        [
            labels,
            augmented_labels,
        ],
        axis=0,
    )

    return (
        combined_matrix,
        combined_labels,
        augmented_count,
    )

In [6]:
with open(
    PRIMARY_CONTRACT_PATH,
    "r",
    encoding="utf-8",
) as file:
    primary_contract = json.load(
        file
    )

with open(
    STAGE_05A_GATE_PATH,
    "r",
    encoding="utf-8",
) as file:
    stage_05a_gate = json.load(
        file
    )

with open(
    BASELINE_GATE_PATH,
    "r",
    encoding="utf-8",
) as file:
    stage_05b_gate = json.load(
        file
    )

with open(
    STAGE_06C_GATE_PATH,
    "r",
    encoding="utf-8",
) as file:
    stage_06c_gate = json.load(
        file
    )

with open(
    STAGE_06C_CONFIG_PATH,
    "r",
    encoding="utf-8",
) as file:
    stage_06c_config = json.load(
        file
    )

input_audit = {
    "05A_contract_passed": bool(
        stage_05a_gate.get(
            "overall_raw_feature_contract_passed",
            False,
        )
    ),
    "05B_grouped_CV_passed": bool(
        stage_05b_gate.get(
            "overall_raw_RF_grouped_CV_passed",
            False,
        )
    ),
    "06C_representation_amplification_passed": bool(
        stage_06c_gate.get(
            "overall_representation_amplification_stage_passed",
            False,
        )
    ),
    "06C_official_test_unused": bool(
        stage_06c_gate.get(
            "official_test_unused",
            False,
        )
    ),
    "raw8_matches_05A_contract": (
        RAW8_FEATURES
        == primary_contract.get(
            "input_features",
            [],
        )
    ),
    "06C_active_features_match": (
        stage_06c_config.get(
            "active_features"
        )
        == CORE6_FEATURES
    ),
    "official_test_unused_in_07A": True,
}

print(
    json.dumps(
        input_audit,
        indent=2,
    )
)

if not all(
    input_audit.values()
):
    raise RuntimeError(
        "Input audit failed."
    )

csv_format = detect_csv_format(
    TRAIN_PATH
)

train = pd.read_csv(
    TRAIN_PATH,
    sep=csv_format[
        "sep"
    ],
    encoding=csv_format[
        "encoding"
    ],
    low_memory=False,
)

train[
    "ORIGINAL_ROW_ID"
] = np.arange(
    len(
        train
    ),
    dtype=np.int64,
)

fold_assignment_raw = pd.read_csv(
    FOLD_ASSIGNMENT_PATH
)

well_column = [
    column
    for column in fold_assignment_raw.columns
    if column.upper()
    == "WELL"
][0]

fold_column = [
    column
    for column in fold_assignment_raw.columns
    if (
        "FOLD"
        in column.upper()
        and "SCORE"
        not in column.upper()
    )
][0]

fold_assignment = (
    fold_assignment_raw[
        [
            well_column,
            fold_column,
        ]
    ]
    .rename(
        columns={
            well_column: "WELL",
            fold_column: "GROUP_FOLD",
        }
    )
    .drop_duplicates(
        subset=[
            "WELL"
        ]
    )
)

fold_assignment[
    "GROUP_FOLD"
] = pd.to_numeric(
    fold_assignment[
        "GROUP_FOLD"
    ],
    errors="raise",
).astype(int)

train[
    "GROUP_FOLD"
] = train[
    "WELL"
].map(
    dict(
        zip(
            fold_assignment[
                "WELL"
            ],
            fold_assignment[
                "GROUP_FOLD"
            ],
        )
    )
)

if train[
    "GROUP_FOLD"
].isna().any():
    raise RuntimeError(
        "Wells without grouped fold assignment."
    )

train[
    "GROUP_FOLD"
] = train[
    "GROUP_FOLD"
].astype(int)

TARGET_COLUMN = (
    "FORCE_2020_LITHOFACIES_LITHOLOGY"
)

train[
    "LITHOLOGY_CODE"
] = pd.to_numeric(
    train[
        TARGET_COLUMN
    ],
    errors="coerce",
)

if train[
    "LITHOLOGY_CODE"
].isna().any():
    raise ValueError(
        "Missing or non-numeric labels."
    )

train[
    "LITHOLOGY_CODE"
] = train[
    "LITHOLOGY_CODE"
].astype(
    np.int64
)

unknown_classes = sorted(
    set(
        train[
            "LITHOLOGY_CODE"
        ].unique()
    )
    - set(
        GLOBAL_CLASS_CODES
    )
)

if unknown_classes:
    raise ValueError(
        f"Unknown classes: {unknown_classes}"
    )

clean_features = pd.DataFrame(
    index=train.index
)

for feature in RAW8_FEATURES:
    if feature not in train.columns:
        raise KeyError(
            f"Missing feature: {feature}"
        )

    clean_features[
        feature
    ] = clean_feature_series(
        train,
        feature,
    )

baseline_oof = pd.read_parquet(
    BASELINE_OOF_PATH
)

print("Train shape:", train.shape)
print("Wells:", train["WELL"].nunique())
print("Folds:", sorted(train["GROUP_FOLD"].unique()))
print("Baseline OOF:", baseline_oof.shape)

{
  "05A_contract_passed": true,
  "05B_grouped_CV_passed": true,
  "06C_representation_amplification_passed": true,
  "06C_official_test_unused": true,
  "raw8_matches_05A_contract": true,
  "06C_active_features_match": true,
  "official_test_unused_in_07A": true
}
Train shape: (1170511, 32)
Wells: 98
Folds: [1, 2, 3, 4, 5]
Baseline OOF: (1170511, 30)


# 5. Inner grouped candidate search

In [7]:
def fit_candidate_model(
    training_indices,
    feature_names,
    candidate_specification,
    n_estimators,
    seed,
):
    imputer = SimpleImputer(
        strategy="median"
    )

    raw_matrix = clean_features.loc[
        training_indices,
        feature_names,
    ].to_numpy(
        dtype=np.float32
    )

    labels = train.loc[
        training_indices,
        "LITHOLOGY_CODE",
    ].to_numpy(
        dtype=np.int64
    )

    imputed_matrix = imputer.fit_transform(
        raw_matrix
    ).astype(
        np.float32,
        copy=False,
    )

    (
        fitted_matrix,
        fitted_labels,
        augmented_rows,
    ) = build_augmented_training_matrix(
        imputed_matrix,
        labels,
        feature_names,
        candidate_specification,
        imputer.statistics_,
        seed,
    )

    parameters = dict(
        BASE_RF_PARAMETERS
    )

    parameters[
        "n_estimators"
    ] = int(
        n_estimators
    )

    parameters[
        "random_state"
    ] = int(
        seed
    )

    model = RandomForestClassifier(
        **parameters
    )

    model.fit(
        fitted_matrix,
        fitted_labels,
    )

    return {
        "model": model,
        "imputer": imputer,
        "training_rows_original": int(
            len(
                imputed_matrix
            )
        ),
        "training_rows_augmented": int(
            augmented_rows
        ),
        "training_rows_total": int(
            len(
                fitted_matrix
            )
        ),
    }


def evaluate_clean_and_stress(
    model,
    imputer,
    validation_indices,
    feature_names,
):
    raw_validation = clean_features.loc[
        validation_indices,
        feature_names,
    ].copy()

    labels = train.loc[
        validation_indices,
        "LITHOLOGY_CODE",
    ].to_numpy(
        dtype=np.int64
    )

    wells = train.loc[
        validation_indices,
        "WELL",
    ].to_numpy()

    validation_matrix = imputer.transform(
        raw_validation.to_numpy(
            dtype=np.float32
        )
    ).astype(
        np.float32,
        copy=False,
    )

    clean_probabilities = predict_probabilities_batched(
        model,
        validation_matrix,
    )

    clean_metrics = prediction_metrics(
        labels,
        clean_probabilities,
        wells,
    )

    clean_outputs = row_outputs(
        labels,
        clean_probabilities,
    )

    stress_rows = []

    for stress_feature in [
        "DTC",
        "RHOB",
    ]:
        if stress_feature not in feature_names:
            continue

        eligible = raw_validation[
            stress_feature
        ].notna().to_numpy()

        eligible_count = int(
            eligible.sum()
        )

        if eligible_count == 0:
            raise RuntimeError(
                f"No eligible rows for {stress_feature}."
            )

        intervened_raw = raw_validation.loc[
            eligible
        ].copy()

        intervened_raw[
            stress_feature
        ] = np.nan

        intervened_matrix = imputer.transform(
            intervened_raw.to_numpy(
                dtype=np.float32
            )
        ).astype(
            np.float32,
            copy=False,
        )

        intervened_probabilities = predict_probabilities_batched(
            model,
            intervened_matrix,
        )

        eligible_labels = labels[
            eligible
        ]

        eligible_wells = wells[
            eligible
        ]

        intervention_outputs = row_outputs(
            eligible_labels,
            intervened_probabilities,
        )

        control_loss = clean_outputs[
            "row_log_loss"
        ][
            eligible
        ]

        delta_log_loss = (
            intervention_outputs[
                "row_log_loss"
            ]
            - control_loss
        )

        control_correct = clean_outputs[
            "correct"
        ][
            eligible
        ].astype(float)

        intervention_correct = intervention_outputs[
            "correct"
        ].astype(float)

        stress_rows.append({
            "stress_feature": stress_feature,
            "eligible_rows": eligible_count,
            "eligible_wells": int(
                pd.Series(
                    eligible_wells
                ).nunique()
            ),
            "pooled_mean_delta_log_loss": float(
                delta_log_loss.mean()
            ),
            "well_balanced_mean_delta_log_loss": well_balanced_mean(
                delta_log_loss,
                eligible_wells,
            ),
            "pooled_accuracy_change": float(
                intervention_correct.mean()
                - control_correct.mean()
            ),
            "well_balanced_accuracy_change": (
                well_balanced_mean(
                    intervention_correct,
                    eligible_wells,
                )
                - well_balanced_mean(
                    control_correct,
                    eligible_wells,
                )
            ),
        })

    stress_metrics = pd.DataFrame(
        stress_rows
    )

    mean_positive_stress_harm = float(
        np.maximum(
            stress_metrics[
                "well_balanced_mean_delta_log_loss"
            ].to_numpy(float),
            0.0,
        ).mean()
    )

    selection_score = (
        clean_metrics[
            "well_balanced_log_loss"
        ]
        + SELECTION_STRESS_WEIGHT
        * mean_positive_stress_harm
    )

    return {
        "clean_probabilities": clean_probabilities,
        "clean_metrics": clean_metrics,
        "stress_metrics": stress_metrics,
        "mean_positive_stress_harm": mean_positive_stress_harm,
        "selection_score": float(
            selection_score
        ),
    }


inner_metric_rows = []
outer_selection_rows = []

for outer_fold in ACTIVE_OUTER_FOLDS:
    print(
        "\n"
        + "=" * 100
    )

    print(
        "OUTER FOLD:",
        outer_fold,
    )

    inner_search_path = (
        INNER_SEARCH_DIR
        / f"outer_fold_{outer_fold}__inner_candidate_metrics.csv"
    )

    selection_path = (
        INNER_SEARCH_DIR
        / f"outer_fold_{outer_fold}__candidate_selection.json"
    )

    if (
        SKIP_COMPLETED_INNER_SEARCH
        and inner_search_path.exists()
        and selection_path.exists()
    ):
        print(
            "Loading cached inner search..."
        )

        outer_inner_metrics = pd.read_csv(
            inner_search_path
        )

        with open(
            selection_path,
            "r",
            encoding="utf-8",
        ) as file:
            selection_payload = json.load(
                file
            )

        selected_candidate = selection_payload[
            "selected_candidate"
        ]

    else:
        outer_inner_rows = []

        inner_folds = [
            fold
            for fold in ALL_FOLDS
            if fold != outer_fold
        ]

        for inner_validation_fold in inner_folds:
            inner_training_mask = (
                train[
                    "GROUP_FOLD"
                ].isin(
                    [
                        fold
                        for fold in inner_folds
                        if fold
                        != inner_validation_fold
                    ]
                )
            )

            inner_validation_mask = (
                train[
                    "GROUP_FOLD"
                ]
                == inner_validation_fold
            )

            training_wells = set(
                train.loc[
                    inner_training_mask,
                    "WELL",
                ].unique()
            )

            validation_wells = set(
                train.loc[
                    inner_validation_mask,
                    "WELL",
                ].unique()
            )

            if training_wells & validation_wells:
                raise RuntimeError(
                    "Inner WELL leakage."
                )

            sampled_training_indices = (
                well_balanced_training_sample(
                    train,
                    inner_training_mask,
                    INNER_TRAINING_ROW_CAP,
                    stable_seed(
                        outer_fold,
                        inner_validation_fold,
                        "inner_training_sample",
                    ),
                )
            )

            validation_indices = train.index[
                inner_validation_mask
            ].to_numpy()

            for candidate_id, candidate_specification in CANDIDATES.items():
                print(
                    "Outer",
                    outer_fold,
                    "| inner validation",
                    inner_validation_fold,
                    "|",
                    candidate_id,
                )

                start_time = time.time()

                feature_names = candidate_specification[
                    "features"
                ]

                fitted = fit_candidate_model(
                    sampled_training_indices,
                    feature_names,
                    candidate_specification,
                    INNER_N_ESTIMATORS,
                    stable_seed(
                        outer_fold,
                        inner_validation_fold,
                        candidate_id,
                        "inner_model",
                    ),
                )

                evaluation = evaluate_clean_and_stress(
                    fitted[
                        "model"
                    ],
                    fitted[
                        "imputer"
                    ],
                    validation_indices,
                    feature_names,
                )

                clean_metrics = evaluation[
                    "clean_metrics"
                ]

                stress_lookup = (
                    evaluation[
                        "stress_metrics"
                    ]
                    .set_index(
                        "stress_feature"
                    )
                )

                row = {
                    "outer_fold": int(
                        outer_fold
                    ),
                    "inner_validation_fold": int(
                        inner_validation_fold
                    ),
                    "candidate_id": candidate_id,
                    "feature_count": int(
                        len(
                            feature_names
                        )
                    ),
                    "training_rows_original": fitted[
                        "training_rows_original"
                    ],
                    "training_rows_augmented": fitted[
                        "training_rows_augmented"
                    ],
                    "training_rows_total": fitted[
                        "training_rows_total"
                    ],
                    "validation_rows": clean_metrics[
                        "rows"
                    ],
                    "validation_wells": clean_metrics[
                        "wells"
                    ],
                    "clean_accuracy": clean_metrics[
                        "accuracy"
                    ],
                    "clean_balanced_accuracy": clean_metrics[
                        "balanced_accuracy"
                    ],
                    "clean_macro_f1": clean_metrics[
                        "macro_f1"
                    ],
                    "clean_log_loss": clean_metrics[
                        "log_loss"
                    ],
                    "clean_well_balanced_accuracy": clean_metrics[
                        "well_balanced_accuracy"
                    ],
                    "clean_well_balanced_log_loss": clean_metrics[
                        "well_balanced_log_loss"
                    ],
                    "DTC_well_balanced_delta_log_loss": float(
                        stress_lookup.loc[
                            "DTC",
                            "well_balanced_mean_delta_log_loss",
                        ]
                    ),
                    "RHOB_well_balanced_delta_log_loss": float(
                        stress_lookup.loc[
                            "RHOB",
                            "well_balanced_mean_delta_log_loss",
                        ]
                    ),
                    "mean_positive_stress_harm": evaluation[
                        "mean_positive_stress_harm"
                    ],
                    "selection_score": evaluation[
                        "selection_score"
                    ],
                    "runtime_seconds": float(
                        time.time()
                        - start_time
                    ),
                    "inner_training_wells_disjoint": True,
                    "outer_validation_fold_unused": True,
                    "official_test_unused": True,
                }

                outer_inner_rows.append(
                    row
                )

                del fitted
                del evaluation
                gc.collect()

        outer_inner_metrics = pd.DataFrame(
            outer_inner_rows
        )

        candidate_aggregate = (
            outer_inner_metrics.groupby(
                "candidate_id",
                as_index=False,
            )
            .agg(
                inner_folds=(
                    "inner_validation_fold",
                    "nunique",
                ),
                mean_selection_score=(
                    "selection_score",
                    "mean",
                ),
                mean_clean_well_balanced_log_loss=(
                    "clean_well_balanced_log_loss",
                    "mean",
                ),
                mean_positive_stress_harm=(
                    "mean_positive_stress_harm",
                    "mean",
                ),
                mean_clean_macro_f1=(
                    "clean_macro_f1",
                    "mean",
                ),
                mean_clean_well_balanced_accuracy=(
                    "clean_well_balanced_accuracy",
                    "mean",
                ),
            )
            .sort_values(
                [
                    "mean_selection_score",
                    "mean_clean_well_balanced_log_loss",
                    "mean_clean_macro_f1",
                    "candidate_id",
                ],
                ascending=[
                    True,
                    True,
                    False,
                    True,
                ],
            )
            .reset_index(
                drop=True
            )
        )

        selected_candidate = candidate_aggregate.loc[
            0,
            "candidate_id",
        ]

        selection_payload = {
            "outer_fold": int(
                outer_fold
            ),
            "selected_candidate": selected_candidate,
            "selection_score": float(
                candidate_aggregate.loc[
                    0,
                    "mean_selection_score",
                ]
            ),
            "selection_rule": (
                "minimum mean inner selection score; "
                "tie-break minimum clean well-balanced log loss; "
                "then maximum macro F1; then candidate_id"
            ),
            "outer_validation_fold_unused": True,
            "official_test_unused": True,
            "candidate_ranking": candidate_aggregate.to_dict(
                orient="records"
            ),
        }

        outer_inner_metrics.to_csv(
            inner_search_path,
            index=False,
        )

        with open(
            selection_path,
            "w",
            encoding="utf-8",
        ) as file:
            json.dump(
                json_safe(
                    selection_payload
                ),
                file,
                ensure_ascii=False,
                indent=2,
            )

    inner_metric_rows.extend(
        outer_inner_metrics.to_dict(
            orient="records"
        )
    )

    selected_aggregate = (
        outer_inner_metrics.groupby(
            "candidate_id",
            as_index=False,
        )
        .agg(
            mean_selection_score=(
                "selection_score",
                "mean",
            ),
            mean_clean_well_balanced_log_loss=(
                "clean_well_balanced_log_loss",
                "mean",
            ),
            mean_positive_stress_harm=(
                "mean_positive_stress_harm",
                "mean",
            ),
            mean_clean_macro_f1=(
                "clean_macro_f1",
                "mean",
            ),
        )
    )

    selected_row = selected_aggregate.loc[
        selected_aggregate[
            "candidate_id"
        ]
        == selected_candidate
    ].iloc[0]

    outer_selection_rows.append({
        "outer_fold": int(
            outer_fold
        ),
        "selected_candidate": selected_candidate,
        "mean_inner_selection_score": float(
            selected_row[
                "mean_selection_score"
            ]
        ),
        "mean_inner_clean_well_balanced_log_loss": float(
            selected_row[
                "mean_clean_well_balanced_log_loss"
            ]
        ),
        "mean_inner_positive_stress_harm": float(
            selected_row[
                "mean_positive_stress_harm"
            ]
        ),
        "mean_inner_clean_macro_f1": float(
            selected_row[
                "mean_clean_macro_f1"
            ]
        ),
        "outer_validation_fold_unused_during_selection": True,
    })

inner_metrics = pd.DataFrame(
    inner_metric_rows
)

outer_selections = pd.DataFrame(
    outer_selection_rows
)

display(
    outer_selections
)

display(
    inner_metrics.groupby(
        "candidate_id",
        as_index=False,
    ).agg(
        mean_selection_score=(
            "selection_score",
            "mean",
        ),
        mean_clean_well_balanced_log_loss=(
            "clean_well_balanced_log_loss",
            "mean",
        ),
        mean_positive_stress_harm=(
            "mean_positive_stress_harm",
            "mean",
        ),
        mean_clean_macro_f1=(
            "clean_macro_f1",
            "mean",
        ),
    ).sort_values(
        "mean_selection_score"
    )
)


OUTER FOLD: 1
Outer 1 | inner validation 2 | regularized_raw8
Outer 1 | inner validation 2 | regularized_core6
Outer 1 | inner validation 2 | global_single_log_dropout_raw8
Outer 1 | inner validation 2 | targeted_DTC_RHOB_dropout_raw8
Outer 1 | inner validation 3 | regularized_raw8
Outer 1 | inner validation 3 | regularized_core6
Outer 1 | inner validation 3 | global_single_log_dropout_raw8
Outer 1 | inner validation 3 | targeted_DTC_RHOB_dropout_raw8
Outer 1 | inner validation 4 | regularized_raw8
Outer 1 | inner validation 4 | regularized_core6
Outer 1 | inner validation 4 | global_single_log_dropout_raw8
Outer 1 | inner validation 4 | targeted_DTC_RHOB_dropout_raw8
Outer 1 | inner validation 5 | regularized_raw8
Outer 1 | inner validation 5 | regularized_core6
Outer 1 | inner validation 5 | global_single_log_dropout_raw8
Outer 1 | inner validation 5 | targeted_DTC_RHOB_dropout_raw8

OUTER FOLD: 2
Outer 2 | inner validation 1 | regularized_raw8
Outer 2 | inner validation 1 | regular

,outer_fold,selected_candidate,mean_inner_selection_score,mean_inner_clean_well_balanced_log_loss,mean_inner_positive_stress_harm,mean_inner_clean_macro_f1,outer_validation_fold_unused_during_selection
0,1,targeted_DTC_RHOB_dropout_raw8,1.057144,1.030367,0.053553,0.284432,True
1,2,global_single_log_dropout_raw8,0.988440,0.949943,0.076994,0.309548,True
2,3,global_single_log_dropout_raw8,1.042918,1.004782,0.076271,0.361086,True
3,4,targeted_DTC_RHOB_dropout_raw8,1.077422,1.045747,0.063350,0.304326,True
4,5,global_single_log_dropout_raw8,1.044094,1.011158,0.065873,0.347022,True


,candidate_id,mean_selection_score,mean_clean_well_balanced_log_loss,mean_positive_stress_harm,mean_clean_macro_f1
3,targeted_DTC_RHOB_dropout_raw8,1.043584,1.016117,0.054935,0.311171
0,global_single_log_dropout_raw8,1.045805,1.003790,0.084031,0.324057
2,regularized_raw8,1.090711,1.019750,0.141923,0.320572
1,regularized_core6,1.113008,1.037645,0.150726,0.307733


In [8]:
outer_clean_metric_rows = []
outer_stress_metric_rows = []
artifact_rows = []
importance_rows = []
outer_oof_parts = []

for selection_row in outer_selections.itertuples(
    index=False
):
    outer_fold = int(
        selection_row.outer_fold
    )

    selected_candidate = (
        selection_row.selected_candidate
    )

    candidate_specification = CANDIDATES[
        selected_candidate
    ]

    feature_names = candidate_specification[
        "features"
    ]

    prefix = (
        f"outer_fold_{outer_fold}"
        f"__{selected_candidate}"
    )

    model_path = (
        MODEL_DIR
        / f"{prefix}__RF.joblib"
    )

    imputer_path = (
        MODEL_DIR
        / f"{prefix}__median_imputer.joblib"
    )

    metadata_path = (
        MODEL_DIR
        / f"{prefix}__metadata.json"
    )

    oof_path = (
        OUTER_OOF_DIR
        / f"{prefix}__OOF.parquet"
    )

    cache_complete = all(
        path.exists()
        for path in [
            model_path,
            imputer_path,
            metadata_path,
            oof_path,
        ]
    )

    if (
        SKIP_COMPLETED_OUTER_FOLDS
        and cache_complete
    ):
        print(
            "Loading cached outer fold:",
            outer_fold,
        )

        outer_oof = pd.read_parquet(
            oof_path
        )

        with open(
            metadata_path,
            "r",
            encoding="utf-8",
        ) as file:
            metadata = json.load(
                file
            )

        if not metadata.get(
            "outer_fold_gate_passed",
            False,
        ):
            raise RuntimeError(
                f"Cached outer fold failed: {outer_fold}"
            )

        outer_oof_parts.append(
            outer_oof
        )

        outer_clean_metric_rows.append(
            metadata[
                "clean_metrics"
            ]
        )

        outer_stress_metric_rows.extend(
            metadata[
                "stress_metrics"
            ]
        )

        importance_rows.extend(
            metadata[
                "feature_importances"
            ]
        )

        for artifact_type, path in [
            (
                "random_forest",
                model_path,
            ),
            (
                "median_imputer",
                imputer_path,
            ),
            (
                "outer_oof",
                oof_path,
            ),
        ]:
            artifact_rows.append({
                "outer_fold": outer_fold,
                "selected_candidate": selected_candidate,
                "artifact_type": artifact_type,
                "path": str(
                    path
                ),
                "sha256": sha256_file(
                    path
                ),
                "status": "loaded_from_cache",
            })

        continue

    print(
        "\nFitting selected outer candidate:",
        outer_fold,
        selected_candidate,
    )

    start_time = time.time()

    validation_mask = (
        train[
            "GROUP_FOLD"
        ]
        == outer_fold
    )

    training_mask = ~validation_mask

    training_wells = set(
        train.loc[
            training_mask,
            "WELL",
        ].unique()
    )

    validation_wells = set(
        train.loc[
            validation_mask,
            "WELL",
        ].unique()
    )

    if training_wells & validation_wells:
        raise RuntimeError(
            "Outer WELL leakage."
        )

    training_indices = train.index[
        training_mask
    ].to_numpy()

    validation_indices = train.index[
        validation_mask
    ].to_numpy()

    fitted = fit_candidate_model(
        training_indices,
        feature_names,
        candidate_specification,
        FINAL_N_ESTIMATORS,
        stable_seed(
            outer_fold,
            selected_candidate,
            "final_outer_model",
        ),
    )

    evaluation = evaluate_clean_and_stress(
        fitted[
            "model"
        ],
        fitted[
            "imputer"
        ],
        validation_indices,
        feature_names,
    )

    probabilities = evaluation[
        "clean_probabilities"
    ]

    labels = train.loc[
        validation_indices,
        "LITHOLOGY_CODE",
    ].to_numpy(
        dtype=np.int64
    )

    outputs = row_outputs(
        labels,
        probabilities,
    )

    outer_oof = train.loc[
        validation_indices,
        [
            "ORIGINAL_ROW_ID",
            "WELL",
            "GROUP_FOLD",
            "LITHOLOGY_CODE",
        ],
    ].copy()

    outer_oof = outer_oof.rename(
        columns={
            "GROUP_FOLD": "outer_fold",
            "LITHOLOGY_CODE": "true_code",
        }
    )

    outer_oof[
        "selected_candidate"
    ] = selected_candidate

    outer_oof[
        "feature_count"
    ] = len(
        feature_names
    )

    outer_oof[
        "predicted_code"
    ] = outputs[
        "predicted"
    ]

    outer_oof[
        "correct"
    ] = outputs[
        "correct"
    ]

    outer_oof[
        "true_class_probability"
    ] = outputs[
        "true_probability"
    ]

    outer_oof[
        "row_log_loss"
    ] = outputs[
        "row_log_loss"
    ]

    for probability_index, class_code in enumerate(
        GLOBAL_CLASS_CODES
    ):
        outer_oof[
            f"prob_{class_code}"
        ] = probabilities[
            :,
            probability_index,
        ].astype(
            np.float32
        )

    clean_metrics = dict(
        evaluation[
            "clean_metrics"
        ]
    )

    clean_metrics.update({
        "outer_fold": outer_fold,
        "selected_candidate": selected_candidate,
        "feature_count": len(
            feature_names
        ),
        "training_rows_original": fitted[
            "training_rows_original"
        ],
        "training_rows_augmented": fitted[
            "training_rows_augmented"
        ],
        "training_rows_total": fitted[
            "training_rows_total"
        ],
        "runtime_seconds": float(
            time.time()
            - start_time
        ),
    })

    stress_metrics = evaluation[
        "stress_metrics"
    ].copy()

    stress_metrics[
        "outer_fold"
    ] = outer_fold

    stress_metrics[
        "selected_candidate"
    ] = selected_candidate

    fold_importance_rows = []

    for feature, importance in zip(
        feature_names,
        fitted[
            "model"
        ].feature_importances_,
    ):
        fold_importance_rows.append({
            "outer_fold": outer_fold,
            "selected_candidate": selected_candidate,
            "feature": feature,
            "importance": float(
                importance
            ),
        })

    outer_fold_audit = {
        "outer_fold": outer_fold,
        "selected_candidate": selected_candidate,
        "training_validation_wells_disjoint": True,
        "candidate_selected_without_outer_validation": True,
        "model_fit_on_outer_training_only": True,
        "imputer_fit_on_outer_training_only": True,
        "augmentation_uses_outer_training_medians_only": True,
        "single_log_augmentation": True,
        "labels_unchanged_in_augmentation": True,
        "probabilities_finite": bool(
            np.isfinite(
                probabilities
            ).all()
        ),
        "probability_rows_sum_to_one": bool(
            np.allclose(
                probabilities.sum(
                    axis=1
                ),
                1.0,
                rtol=1e-6,
                atol=1e-6,
            )
        ),
        "row_count_correct": (
            len(
                outer_oof
            )
            == int(
                validation_mask.sum()
            )
        ),
        "row_ids_unique": bool(
            not outer_oof[
                "ORIGINAL_ROW_ID"
            ].duplicated().any()
        ),
        "official_test_unused": True,
    }

    outer_fold_audit[
        "outer_fold_gate_passed"
    ] = bool(
        all(
            value
            for key, value in outer_fold_audit.items()
            if (
                isinstance(
                    value,
                    bool,
                )
                and key
                != "outer_fold_gate_passed"
            )
        )
    )

    if not outer_fold_audit[
        "outer_fold_gate_passed"
    ]:
        raise RuntimeError(
            f"Outer fold gate failed: {outer_fold}"
        )

    joblib.dump(
        fitted[
            "model"
        ],
        model_path,
        compress=JOBLIB_COMPRESSION,
    )

    joblib.dump(
        fitted[
            "imputer"
        ],
        imputer_path,
        compress=JOBLIB_COMPRESSION,
    )

    outer_oof.to_parquet(
        oof_path,
        index=False,
    )

    metadata = {
        **outer_fold_audit,
        "features": feature_names,
        "candidate_specification": candidate_specification,
        "RF_parameters": {
            **BASE_RF_PARAMETERS,
            "n_estimators": FINAL_N_ESTIMATORS,
        },
        "clean_metrics": clean_metrics,
        "stress_metrics": stress_metrics.to_dict(
            orient="records"
        ),
        "feature_importances": fold_importance_rows,
    }

    with open(
        metadata_path,
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            json_safe(
                metadata
            ),
            file,
            ensure_ascii=False,
            indent=2,
        )

    outer_oof_parts.append(
        outer_oof
    )

    outer_clean_metric_rows.append(
        clean_metrics
    )

    outer_stress_metric_rows.extend(
        stress_metrics.to_dict(
            orient="records"
        )
    )

    importance_rows.extend(
        fold_importance_rows
    )

    for artifact_type, path in [
        (
            "random_forest",
            model_path,
        ),
        (
            "median_imputer",
            imputer_path,
        ),
        (
            "outer_oof",
            oof_path,
        ),
    ]:
        artifact_rows.append({
            "outer_fold": outer_fold,
            "selected_candidate": selected_candidate,
            "artifact_type": artifact_type,
            "path": str(
                path
            ),
            "sha256": sha256_file(
                path
            ),
            "status": "built",
        })

    del fitted
    del evaluation
    del probabilities
    gc.collect()

outer_oof = (
    pd.concat(
        outer_oof_parts,
        ignore_index=True,
    )
    .sort_values(
        "ORIGINAL_ROW_ID"
    )
    .reset_index(
        drop=True
    )
)

outer_clean_metrics = pd.DataFrame(
    outer_clean_metric_rows
)

outer_stress_metrics = pd.DataFrame(
    outer_stress_metric_rows
)

feature_importances = pd.DataFrame(
    importance_rows
)

artifact_manifest = pd.DataFrame(
    artifact_rows
)

display(
    outer_clean_metrics
)

display(
    outer_stress_metrics
)


Fitting selected outer candidate: 1 targeted_DTC_RHOB_dropout_raw8

Fitting selected outer candidate: 2 global_single_log_dropout_raw8

Fitting selected outer candidate: 3 global_single_log_dropout_raw8

Fitting selected outer candidate: 4 targeted_DTC_RHOB_dropout_raw8

Fitting selected outer candidate: 5 global_single_log_dropout_raw8


,rows,wells,accuracy,balanced_accuracy,macro_f1,log_loss,well_balanced_accuracy,well_balanced_log_loss,worst_well_log_loss,outer_fold,selected_candidate,feature_count,training_rows_original,training_rows_augmented,training_rows_total,runtime_seconds
0,232674,20,0.724279,0.406305,0.378416,0.909357,0.719970,0.900797,2.225389,1,targeted_DTC_RHOB_dropout_raw8,8,937837,187567,1125404,237.451269
1,236205,20,0.670392,0.357515,0.339594,1.000596,0.656423,1.038380,1.729792,2,global_single_log_dropout_raw8,8,934306,186861,1121167,233.082075
2,230578,21,0.661884,0.328401,0.254039,1.005413,0.711070,0.913080,1.766218,3,global_single_log_dropout_raw8,8,939933,187987,1127920,242.899380
3,237132,17,0.605755,0.348239,0.314287,1.096238,0.602542,1.094730,1.755292,4,targeted_DTC_RHOB_dropout_raw8,8,933379,186676,1120055,237.414764
4,233922,20,0.702431,0.318504,0.281544,0.899329,0.675719,0.919477,1.398205,5,global_single_log_dropout_raw8,8,936589,187318,1123907,252.555485


,stress_feature,eligible_rows,eligible_wells,pooled_mean_delta_log_loss,well_balanced_mean_delta_log_loss,pooled_accuracy_change,well_balanced_accuracy_change,outer_fold,selected_candidate
0,DTC,216573,20,0.048608,0.040777,-0.008265,-0.010784,1,targeted_DTC_RHOB_dropout_raw8
1,RHOB,199559,20,0.020492,0.036234,-0.002250,-0.015608,1,targeted_DTC_RHOB_dropout_raw8
2,DTC,223821,20,0.032785,0.021588,0.005227,0.000367,2,global_single_log_dropout_raw8
3,RHOB,192466,20,0.072728,0.076586,-0.029699,-0.034689,2,global_single_log_dropout_raw8
4,DTC,219988,21,0.056598,0.058039,-0.004755,-0.020676,3,global_single_log_dropout_raw8
5,RHOB,192229,21,0.043938,0.067496,-0.009296,-0.015681,3,global_single_log_dropout_raw8
6,DTC,197988,17,0.046681,0.041365,-0.011597,-0.007725,4,targeted_DTC_RHOB_dropout_raw8
7,RHOB,207944,17,0.044294,0.049808,0.004049,0.003983,4,targeted_DTC_RHOB_dropout_raw8
8,DTC,231278,20,0.096519,0.076753,-0.021550,-0.001958,5,global_single_log_dropout_raw8
9,RHOB,217044,20,0.055095,0.069155,-0.015845,-0.021851,5,global_single_log_dropout_raw8


In [9]:
active_mask = train[
    "GROUP_FOLD"
].isin(
    ACTIVE_OUTER_FOLDS
)

expected_row_ids = set(
    train.loc[
        active_mask,
        "ORIGINAL_ROW_ID",
    ].astype(int)
)

observed_row_ids = set(
    outer_oof[
        "ORIGINAL_ROW_ID"
    ].astype(int)
)

if observed_row_ids != expected_row_ids:
    raise RuntimeError(
        "Selected robust OOF row-id set mismatch."
    )

oof_metrics = prediction_metrics(
    outer_oof[
        "true_code"
    ].to_numpy(
        dtype=np.int64
    ),
    outer_oof[
        PROBABILITY_COLUMNS
    ].to_numpy(
        dtype=np.float64
    ),
    outer_oof[
        "WELL"
    ].to_numpy(),
)

per_well_metrics = []

for (
    well,
    outer_fold,
), frame in outer_oof.groupby(
    [
        "WELL",
        "outer_fold",
    ]
):
    per_well_metrics.append({
        "WELL": well,
        "outer_fold": int(
            outer_fold
        ),
        "rows": int(
            len(
                frame
            )
        ),
        "selected_candidate": frame[
            "selected_candidate"
        ].iloc[0],
        "accuracy": float(
            frame[
                "correct"
            ].mean()
        ),
        "log_loss": float(
            frame[
                "row_log_loss"
            ].mean()
        ),
        "mean_true_class_probability": float(
            frame[
                "true_class_probability"
            ].mean()
        ),
    })

per_well_metrics = pd.DataFrame(
    per_well_metrics
)

well_balanced_metrics = pd.DataFrame([
    {
        "run_mode": RUN_MODE,
        "outer_folds": len(
            ACTIVE_OUTER_FOLDS
        ),
        "wells": int(
            per_well_metrics[
                "WELL"
            ].nunique()
        ),
        "well_balanced_accuracy": float(
            per_well_metrics[
                "accuracy"
            ].mean()
        ),
        "well_balanced_log_loss": float(
            per_well_metrics[
                "log_loss"
            ].mean()
        ),
        "worst_well_log_loss": float(
            per_well_metrics[
                "log_loss"
            ].max()
        ),
    }
])

pooled_metrics = pd.DataFrame([
    {
        "run_mode": RUN_MODE,
        **oof_metrics,
    }
])

baseline_subset = baseline_oof.loc[
    baseline_oof[
        "outer_fold"
    ].isin(
        ACTIVE_OUTER_FOLDS
    ),
    [
        "ORIGINAL_ROW_ID",
        "WELL",
        "outer_fold",
        "true_code",
        "correct",
        "true_class_probability",
        "row_log_loss",
    ],
].copy()

baseline_subset = baseline_subset.rename(
    columns={
        "correct": "baseline_correct",
        "true_class_probability": "baseline_true_class_probability",
        "row_log_loss": "baseline_row_log_loss",
        "true_code": "baseline_true_code",
        "WELL": "baseline_WELL",
        "outer_fold": "baseline_outer_fold",
    }
)

paired_baseline = outer_oof[
    [
        "ORIGINAL_ROW_ID",
        "WELL",
        "outer_fold",
        "true_code",
        "selected_candidate",
        "correct",
        "true_class_probability",
        "row_log_loss",
    ]
].merge(
    baseline_subset,
    on="ORIGINAL_ROW_ID",
    how="inner",
    validate="one_to_one",
)

baseline_pairing_audit = {
    "paired_rows": int(
        len(
            paired_baseline
        )
    ),
    "expected_rows": int(
        len(
            outer_oof
        )
    ),
    "row_count_correct": (
        len(
            paired_baseline
        )
        == len(
            outer_oof
        )
    ),
    "labels_match": bool(
        (
            paired_baseline[
                "true_code"
            ]
            == paired_baseline[
                "baseline_true_code"
            ]
        ).all()
    ),
    "wells_match": bool(
        (
            paired_baseline[
                "WELL"
            ]
            == paired_baseline[
                "baseline_WELL"
            ]
        ).all()
    ),
    "folds_match": bool(
        (
            paired_baseline[
                "outer_fold"
            ]
            == paired_baseline[
                "baseline_outer_fold"
            ]
        ).all()
    ),
}

if not all(
    value
    for key, value in baseline_pairing_audit.items()
    if isinstance(
        value,
        bool,
    )
):
    raise RuntimeError(
        "Baseline pairing audit failed."
    )

paired_baseline[
    "delta_log_loss_robust_minus_reference"
] = (
    paired_baseline[
        "row_log_loss"
    ]
    - paired_baseline[
        "baseline_row_log_loss"
    ]
)

paired_baseline[
    "delta_true_probability_robust_minus_reference"
] = (
    paired_baseline[
        "true_class_probability"
    ]
    - paired_baseline[
        "baseline_true_class_probability"
    ]
)

paired_baseline[
    "accuracy_difference_robust_minus_reference"
] = (
    paired_baseline[
        "correct"
    ].astype(int)
    - paired_baseline[
        "baseline_correct"
    ].astype(int)
)

paired_baseline_per_well = (
    paired_baseline.groupby(
        [
            "WELL",
            "outer_fold",
        ],
        as_index=False,
    )
    .agg(
        rows=(
            "ORIGINAL_ROW_ID",
            "size",
        ),
        selected_candidate=(
            "selected_candidate",
            "first",
        ),
        mean_delta_log_loss_robust_minus_reference=(
            "delta_log_loss_robust_minus_reference",
            "mean",
        ),
        mean_delta_true_probability_robust_minus_reference=(
            "delta_true_probability_robust_minus_reference",
            "mean",
        ),
        accuracy_difference_robust_minus_reference=(
            "accuracy_difference_robust_minus_reference",
            "mean",
        ),
    )
)

baseline_comparison_summary = pd.DataFrame([
    {
        "run_mode": RUN_MODE,
        "rows": int(
            len(
                paired_baseline
            )
        ),
        "wells": int(
            paired_baseline[
                "WELL"
            ].nunique()
        ),
        "pooled_mean_delta_log_loss_robust_minus_reference": float(
            paired_baseline[
                "delta_log_loss_robust_minus_reference"
            ].mean()
        ),
        "well_balanced_mean_delta_log_loss_robust_minus_reference": float(
            paired_baseline_per_well[
                "mean_delta_log_loss_robust_minus_reference"
            ].mean()
        ),
        "well_balanced_accuracy_difference_robust_minus_reference": float(
            paired_baseline_per_well[
                "accuracy_difference_robust_minus_reference"
            ].mean()
        ),
        "well_balanced_true_probability_difference_robust_minus_reference": float(
            paired_baseline_per_well[
                "mean_delta_true_probability_robust_minus_reference"
            ].mean()
        ),
    }
])

display(
    pooled_metrics
)

display(
    well_balanced_metrics
)

display(
    baseline_comparison_summary
)

,run_mode,rows,wells,accuracy,balanced_accuracy,macro_f1,log_loss,well_balanced_accuracy,well_balanced_log_loss,worst_well_log_loss
0,full,1170511,98,0.672736,0.34349,0.350115,0.982547,0.675693,0.968961,2.225389


,run_mode,outer_folds,wells,well_balanced_accuracy,well_balanced_log_loss,worst_well_log_loss
0,full,5,98,0.675693,0.968961,2.225389


,run_mode,rows,wells,pooled_mean_delta_log_loss_robust_minus_reference,well_balanced_mean_delta_log_loss_robust_minus_reference,well_balanced_accuracy_difference_robust_minus_reference,well_balanced_true_probability_difference_robust_minus_reference
0,full,1170511,98,0.042173,0.03864,-0.012516,-0.049723


# 8. Stage gate

In [10]:
expected_inner_rows = (
    len(
        ACTIVE_OUTER_FOLDS
    )
    * 4
    * len(
        CANDIDATES
    )
)

expected_outer_rows = int(
    active_mask.sum()
)

stage_gate = {
    "official_test_unused": True,
    "05A_contract_passed": bool(
        input_audit[
            "05A_contract_passed"
        ]
    ),
    "05B_reference_passed": bool(
        input_audit[
            "05B_grouped_CV_passed"
        ]
    ),
    "06C_diagnostic_stage_passed": bool(
        input_audit[
            "06C_representation_amplification_passed"
        ]
    ),
    "strict_nested_grouped_candidate_selection": True,
    "outer_validation_unused_during_selection": bool(
        outer_selections[
            "outer_validation_fold_unused_during_selection"
        ].all()
    ),
    "expected_inner_candidate_rows": int(
        expected_inner_rows
    ),
    "observed_inner_candidate_rows": int(
        len(
            inner_metrics
        )
    ),
    "inner_candidate_row_count_correct": (
        len(
            inner_metrics
        )
        == expected_inner_rows
    ),
    "all_inner_training_wells_disjoint": bool(
        inner_metrics[
            "inner_training_wells_disjoint"
        ].all()
    ),
    "all_inner_outer_validation_folds_unused": bool(
        inner_metrics[
            "outer_validation_fold_unused"
        ].all()
    ),
    "expected_outer_fold_count": int(
        len(
            ACTIVE_OUTER_FOLDS
        )
    ),
    "observed_outer_fold_count": int(
        outer_oof[
            "outer_fold"
        ].nunique()
    ),
    "outer_fold_count_correct": (
        outer_oof[
            "outer_fold"
        ].nunique()
        == len(
            ACTIVE_OUTER_FOLDS
        )
    ),
    "expected_outer_OOF_rows": expected_outer_rows,
    "observed_outer_OOF_rows": int(
        len(
            outer_oof
        )
    ),
    "outer_OOF_row_count_correct": (
        len(
            outer_oof
        )
        == expected_outer_rows
    ),
    "outer_OOF_row_ids_exact": (
        observed_row_ids
        == expected_row_ids
    ),
    "outer_OOF_row_ids_unique": bool(
        not outer_oof[
            "ORIGINAL_ROW_ID"
        ].duplicated().any()
    ),
    "outer_probabilities_finite": bool(
        np.isfinite(
            outer_oof[
                PROBABILITY_COLUMNS
            ].to_numpy(float)
        ).all()
    ),
    "outer_probabilities_sum_to_one": bool(
        np.allclose(
            outer_oof[
                PROBABILITY_COLUMNS
            ].to_numpy(float).sum(
                axis=1
            ),
            1.0,
            rtol=1e-6,
            atol=1e-6,
        )
    ),
    "baseline_pairing_complete": bool(
        baseline_pairing_audit[
            "row_count_correct"
        ]
        and baseline_pairing_audit[
            "labels_match"
        ]
        and baseline_pairing_audit[
            "wells_match"
        ]
        and baseline_pairing_audit[
            "folds_match"
        ]
    ),
    "single_log_training_augmentation": True,
    "augmentation_labels_unchanged": True,
    "augmentation_statistics_training_only": True,
    "raw_uncalibrated_probabilities_saved": True,
    "run_mode": RUN_MODE,
}

stage_gate[
    "overall_robust_RF_nested_candidate_selection_passed"
] = bool(
    all(
        value
        for key, value in stage_gate.items()
        if (
            isinstance(
                value,
                bool,
            )
            and key
            != "overall_robust_RF_nested_candidate_selection_passed"
        )
    )
)

print(
    json.dumps(
        stage_gate,
        indent=2,
    )
)

if not stage_gate[
    "overall_robust_RF_nested_candidate_selection_passed"
]:
    raise RuntimeError(
        "07A stage gate failed."
    )

{
  "official_test_unused": true,
  "05A_contract_passed": true,
  "05B_reference_passed": true,
  "06C_diagnostic_stage_passed": true,
  "strict_nested_grouped_candidate_selection": true,
  "outer_validation_unused_during_selection": true,
  "expected_inner_candidate_rows": 80,
  "observed_inner_candidate_rows": 80,
  "inner_candidate_row_count_correct": true,
  "all_inner_training_wells_disjoint": true,
  "all_inner_outer_validation_folds_unused": true,
  "expected_outer_fold_count": 5,
  "observed_outer_fold_count": 5,
  "outer_fold_count_correct": true,
  "expected_outer_OOF_rows": 1170511,
  "observed_outer_OOF_rows": 1170511,
  "outer_OOF_row_count_correct": true,
  "outer_OOF_row_ids_exact": true,
  "outer_OOF_row_ids_unique": true,
  "outer_probabilities_finite": true,
  "outer_probabilities_sum_to_one": true,
  "baseline_pairing_complete": true,
  "single_log_training_augmentation": true,
  "augmentation_labels_unchanged": true,
  "augmentation_statistics_training_only": true,

In [11]:
input_audit_df = pd.DataFrame([
    {
        "audit_item": key,
        "passed": bool(
            value
        ),
    }
    for key, value in input_audit.items()
])

input_audit_df.to_csv(
    OUTPUT_DIR
    / "00_input_audit.csv",
    index=False,
)

candidate_registry.to_csv(
    OUTPUT_DIR
    / "01_candidate_registry.csv",
    index=False,
)

inner_metrics.to_csv(
    OUTPUT_DIR
    / "02_inner_fold_candidate_metrics.csv",
    index=False,
)

outer_selections.to_csv(
    OUTPUT_DIR
    / "03_outer_fold_candidate_selection.csv",
    index=False,
)

outer_clean_metrics.to_csv(
    OUTPUT_DIR
    / "04_outer_fold_clean_metrics.csv",
    index=False,
)

outer_stress_metrics.to_csv(
    OUTPUT_DIR
    / "05_outer_fold_DTC_RHOB_outage_metrics.csv",
    index=False,
)

pooled_metrics.to_csv(
    OUTPUT_DIR
    / "06_pooled_OOF_metrics.csv",
    index=False,
)

well_balanced_metrics.to_csv(
    OUTPUT_DIR
    / "07_well_balanced_OOF_metrics.csv",
    index=False,
)

per_well_metrics.to_csv(
    OUTPUT_DIR
    / "08_per_well_OOF_metrics.csv",
    index=False,
)

baseline_comparison_summary.to_csv(
    OUTPUT_DIR
    / "09_reference_RF_paired_summary.csv",
    index=False,
)

paired_baseline_per_well.to_csv(
    OUTPUT_DIR
    / "10_reference_RF_per_well_comparison.csv",
    index=False,
)

feature_importances.to_csv(
    OUTPUT_DIR
    / "11_selected_model_feature_importances.csv",
    index=False,
)

artifact_manifest.to_csv(
    OUTPUT_DIR
    / "12_artifact_manifest.csv",
    index=False,
)

outer_oof.to_parquet(
    OUTPUT_DIR
    / "13_OOF_predictions_selected_robust_RF.parquet",
    index=False,
)

with open(
    OUTPUT_DIR
    / "14_stage_gate.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        json_safe(
            stage_gate
        ),
        file,
        ensure_ascii=False,
        indent=2,
    )

run_configuration = {
    "stage": (
        "07A_robust_RF_nested_grouped_candidate_selection"
    ),
    "run_mode": RUN_MODE,
    "official_test_used": False,
    "active_outer_folds": ACTIVE_OUTER_FOLDS,
    "all_folds": ALL_FOLDS,
    "candidate_registry": CANDIDATES,
    "base_RF_parameters": BASE_RF_PARAMETERS,
    "inner_n_estimators": int(
        INNER_N_ESTIMATORS
    ),
    "final_n_estimators": int(
        FINAL_N_ESTIMATORS
    ),
    "inner_training_row_cap": int(
        INNER_TRAINING_ROW_CAP
    ),
    "inner_training_sampling": (
        "approximately equal row cap per WELL, "
        "with missing classes restored"
    ),
    "augmentation_fraction": float(
        AUGMENTATION_FRACTION
    ),
    "augmentation_definition": (
        "one selected training log is replaced by the "
        "training-fold median; original row and label are retained"
    ),
    "selection_score": (
        "clean well-balanced log loss + 0.5 × mean positive "
        "DTC/RHOB outage well-balanced delta log loss"
    ),
    "inner_stress_definition": (
        "among originally observed eligible validation rows, "
        "set one of DTC or RHOB to missing and apply the frozen "
        "inner-training median imputer"
    ),
    "outer_evaluation": (
        "clean grouped OOF plus DTC/RHOB outage diagnostics"
    ),
    "probability_status": (
        "raw uncalibrated probabilities"
    ),
    "model_refit_policy": (
        "selected candidate refit once on complete outer-training rows"
    ),
    "next_stage": (
        "07B_selected_robust_RF_strict_nested_temperature_calibration"
    ),
}

with open(
    OUTPUT_DIR
    / "15_run_configuration.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        json_safe(
            run_configuration
        ),
        file,
        ensure_ascii=False,
        indent=2,
    )

input_hashes = {
    "train_csv": sha256_file(
        TRAIN_PATH
    ),
    "fold_assignment": sha256_file(
        FOLD_ASSIGNMENT_PATH
    ),
    "05A_contract": sha256_file(
        PRIMARY_CONTRACT_PATH
    ),
    "05A_gate": sha256_file(
        STAGE_05A_GATE_PATH
    ),
    "05B_baseline_OOF": sha256_file(
        BASELINE_OOF_PATH
    ),
    "05B_gate": sha256_file(
        BASELINE_GATE_PATH
    ),
    "06C_gate": sha256_file(
        STAGE_06C_GATE_PATH
    ),
    "06C_config": sha256_file(
        STAGE_06C_CONFIG_PATH
    ),
}

with open(
    OUTPUT_DIR
    / "16_input_hashes.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        input_hashes,
        file,
        ensure_ascii=False,
        indent=2,
    )

environment = {
    "python": platform.python_version(),
    "platform": platform.platform(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "sklearn": sklearn.__version__,
    "joblib": joblib.__version__,
}

with open(
    OUTPUT_DIR
    / "17_environment.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        environment,
        file,
        ensure_ascii=False,
        indent=2,
    )

print(
    "Outputs saved in:",
    OUTPUT_DIR,
)

Outputs saved in: <LOCAL_PATH_REDACTED>
